# 02 Funnel Analysis - Olist Transaction Journey

本 notebook 聚焦 Olist 交易链路漏斗：`下单 -> 支付 -> 发货 -> 签收 -> 评价`，并验证一个关键增长假设：物流时效越长，差评率越高。

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

ROOT = Path.cwd()
DB_PATH = ROOT / 'data' / 'raw' / 'olist.sqlite'
SQL_PATHS = [
    ROOT / 'sql' / 'metrics_v0.sql',
    ROOT / 'sql' / 'funnel_analysis.sql',
]

if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing database: {DB_PATH}')

conn = sqlite3.connect(DB_PATH)
for sql_path in SQL_PATHS:
    conn.executescript(sql_path.read_text())

print(f'Connected to: {DB_PATH}')

## 1. Overall Funnel

先看全量订单在五个阶段的绝对量与阶段转化率。

In [ ]:
funnel = pd.read_sql_query(
    '''
    SELECT *
    FROM v_funnel_overall
    ''',
    conn,
)

funnel['stage_conversion'] = funnel['orders'] / funnel['orders'].shift(1)
funnel.loc[0, 'stage_conversion'] = 1.0
funnel['overall_conversion'] = funnel['orders'] / funnel.loc[0, 'orders']
funnel

In [ ]:
ax = sns.barplot(data=funnel, x='stage', y='orders', palette='Blues_d')
ax.set_title('Olist Transaction Funnel')
ax.set_xlabel('Stage')
ax.set_ylabel('Orders')

for idx, row in funnel.iterrows():
    ax.text(idx, row['orders'], f"{int(row['orders']):,}\n{row['overall_conversion']:.1%}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 2. Monthly Funnel Trend

按购买月份看漏斗变化，判断是否存在某一时期支付、履约或评价转化的明显恶化。

In [ ]:
funnel_monthly = pd.read_sql_query(
    '''
    WITH monthly AS (
      SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS month,
        SUM(step_purchase) AS purchase_orders,
        SUM(step_payment) AS payment_orders,
        SUM(step_shipped) AS shipped_orders,
        SUM(step_delivered) AS delivered_orders,
        SUM(step_reviewed) AS reviewed_orders
      FROM v_funnel_order
      WHERE order_purchase_timestamp IS NOT NULL
      GROUP BY 1
    )
    SELECT
      month,
      purchase_orders,
      payment_orders * 1.0 / NULLIF(purchase_orders, 0) AS purchase_to_payment,
      shipped_orders * 1.0 / NULLIF(payment_orders, 0) AS payment_to_ship,
      delivered_orders * 1.0 / NULLIF(shipped_orders, 0) AS ship_to_delivered,
      reviewed_orders * 1.0 / NULLIF(delivered_orders, 0) AS delivered_to_review
    FROM monthly
    ORDER BY month
    ''',
    conn,
)

funnel_monthly.tail()

In [ ]:
monthly_rates = funnel_monthly.melt(
    id_vars='month',
    value_vars=['purchase_to_payment', 'payment_to_ship', 'ship_to_delivered', 'delivered_to_review'],
    var_name='metric',
    value_name='rate'
)

ax = sns.lineplot(data=monthly_rates, x='month', y='rate', hue='metric', marker='o')
ax.set_title('Monthly Stage Conversion Rates')
ax.set_xlabel('Purchase Month')
ax.set_ylabel('Conversion Rate')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 3. Hypothesis Check: Logistics Delay vs Bad Review

把已签收且已评价订单按物流时长分桶，观察差评率是否随时长上升。

In [ ]:
delay_bins = pd.read_sql_query(
    '''
    WITH base AS (
      SELECT
        order_id,
        purchase_to_delivered_days,
        delivery_delay_vs_estimate_days,
        review_score,
        is_bad_review,
        CASE
          WHEN purchase_to_delivered_days < 3 THEN '0-3d'
          WHEN purchase_to_delivered_days < 7 THEN '3-7d'
          WHEN purchase_to_delivered_days < 14 THEN '7-14d'
          WHEN purchase_to_delivered_days < 21 THEN '14-21d'
          ELSE '21d+'
        END AS delivery_speed_bucket,
        CASE
          WHEN delivery_delay_vs_estimate_days <= 0 THEN 'On time or early'
          WHEN delivery_delay_vs_estimate_days <= 3 THEN '1-3d late'
          WHEN delivery_delay_vs_estimate_days <= 7 THEN '4-7d late'
          ELSE '7d+ late'
        END AS estimate_gap_bucket
      FROM v_funnel_order
      WHERE step_delivered = 1
        AND step_reviewed = 1
        AND purchase_to_delivered_days IS NOT NULL
        AND review_score IS NOT NULL
    )
    SELECT
      delivery_speed_bucket,
      COUNT(*) AS orders,
      AVG(review_score) AS avg_review_score,
      AVG(is_bad_review) AS bad_review_rate
    FROM base
    GROUP BY 1
    ORDER BY CASE delivery_speed_bucket
      WHEN '0-3d' THEN 1
      WHEN '3-7d' THEN 2
      WHEN '7-14d' THEN 3
      WHEN '14-21d' THEN 4
      ELSE 5
    END
    ''',
    conn,
)

delay_bins

In [ ]:
ax = sns.barplot(data=delay_bins, x='delivery_speed_bucket', y='bad_review_rate', palette='flare')
ax.set_title('Bad Review Rate by Delivery Duration')
ax.set_xlabel('Purchase to Delivered')
ax.set_ylabel('Bad Review Rate')

for idx, row in delay_bins.iterrows():
    ax.text(idx, row['bad_review_rate'], f"{row['bad_review_rate']:.1%}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
estimate_gap = pd.read_sql_query(
    '''
    WITH base AS (
      SELECT
        CASE
          WHEN delivery_delay_vs_estimate_days <= 0 THEN 'On time or early'
          WHEN delivery_delay_vs_estimate_days <= 3 THEN '1-3d late'
          WHEN delivery_delay_vs_estimate_days <= 7 THEN '4-7d late'
          ELSE '7d+ late'
        END AS estimate_gap_bucket,
        review_score,
        is_bad_review
      FROM v_funnel_order
      WHERE step_delivered = 1
        AND step_reviewed = 1
        AND delivery_delay_vs_estimate_days IS NOT NULL
        AND review_score IS NOT NULL
    )
    SELECT
      estimate_gap_bucket,
      COUNT(*) AS orders,
      AVG(review_score) AS avg_review_score,
      AVG(is_bad_review) AS bad_review_rate
    FROM base
    GROUP BY 1
    ORDER BY CASE estimate_gap_bucket
      WHEN 'On time or early' THEN 1
      WHEN '1-3d late' THEN 2
      WHEN '4-7d late' THEN 3
      ELSE 4
    END
    ''',
    conn,
)

estimate_gap

In [ ]:
ax = sns.barplot(data=estimate_gap, x='estimate_gap_bucket', y='bad_review_rate', palette='magma')
ax.set_title('Bad Review Rate by Delay vs Estimated Delivery')
ax.set_xlabel('Delay Bucket')
ax.set_ylabel('Bad Review Rate')

for idx, row in estimate_gap.iterrows():
    ax.text(idx, row['bad_review_rate'], f"{row['bad_review_rate']:.1%}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 4. Interpretation Template

建议在跑出结果后补充三句结论：

- 漏斗最大流失发生在 `X -> Y` 环节，阶段转化率为 `__%`。
- 物流时长从 `A` 提升到 `B` 时，差评率从 `__%` 上升到 `__%`。
- 若假设成立，优先优化晚到订单治理、发货 SLA 和异常订单补偿。